In [5]:

import pandas as pd
import os

os.chdir(r'C:\Users\kotla\Downloads\archivef1')


drivers = pd.read_csv('drivers.csv')
results = pd.read_csv('results.csv')
races = pd.read_csv('races.csv')
constructors = pd.read_csv('constructors.csv')
constructor_results = pd.read_csv('constructor_results.csv')
circuits = pd.read_csv('circuits.csv')
qualifying = pd.read_csv('qualifying.csv')


driver_results = results.merge(races[['raceId','year','name']], on='raceId')
driver_results = driver_results.merge(
    drivers[['driverId','forename','surname','nationality']], on='driverId')
driver_results['driver_name'] = driver_results['forename'] + ' ' + driver_results['surname']
driver_results['is_win'] = driver_results['position'] == '1'
driver_results['is_podium'] = driver_results['position'].isin(['1','2','3'])

driver_summary = driver_results.groupby(['driver_name','nationality']).agg(
    total_races=('raceId','count'),
    total_wins=('is_win','sum'),
    total_podiums=('is_podium','sum'),
    total_points=('points','sum')
).reset_index().sort_values('total_wins', ascending=False)


constructor_year = constructor_results.merge(races[['raceId','year']], on='raceId')
constructor_year = constructor_year.merge(
    constructors[['constructorId','name']], on='constructorId')

constructor_summary = constructor_year.groupby(['name','year']).agg(
    total_points=('points','sum')
).reset_index().sort_values(['year','total_points'], ascending=[True, False])


circuit_map = races.merge(circuits, on='circuitId')
circuit_map = circuit_map.groupby(
    ['circuitId','name_y','location','country','lat','lng']).agg(
    times_hosted=('raceId','count')
).reset_index().rename(columns={'name_y':'circuit_name'})


quali = qualifying.merge(races[['raceId','year','name']], on='raceId')
quali = quali.merge(drivers[['driverId','forename','surname']], on='driverId')
quali['driver_name'] = quali['forename'] + ' ' + quali['surname']
quali = quali.merge(constructors[['constructorId','name']], on='constructorId')
quali = quali.rename(columns={'name_x':'race_name','name_y':'team'})
quali = quali[['year','race_name','driver_name','team','position','q1','q2','q3']]


with pd.ExcelWriter('f1_dashboard_data.xlsx', engine='openpyxl') as writer:
    driver_summary.to_excel(writer, sheet_name='Driver Performance', index=False)
    constructor_summary.to_excel(writer, sheet_name='Constructor Performance', index=False)
    circuit_map.to_excel(writer, sheet_name='Circuits Map', index=False)
    quali.to_excel(writer, sheet_name='Qualifying', index=False)

print(" Done! f1_dashboard_data.xlsx is ready!")

 Done! f1_dashboard_data.xlsx is ready!
